<a href="https://colab.research.google.com/github/nisal-eng/Statistical-Learning-e22206/blob/main/Bayesian_Inference_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **1. Item Response Theory (2PL IRT Model)**

Model Mechanics & Visual InterpretationIn the 2PL Item Response Theory model, the probability of a correct answer $Y_i = 1$ given latent ability $\theta$ is:$$P(Y_i = 1 \mid \theta) = \frac{1}{1 + e^{-a_i(\theta - b_i)}}$$Difficulty Parameter ($b_i$): Shifting $b_i$ horizontally translates the Item Characteristic Curve along the $\theta$-axis. The value $b_i$ corresponds to the exact ability level where a user has a 50% probability of answering correctly ($P(Y_i=1 \mid \theta = b_i) = 0.5$). Increasing $b_i$ shifts the curve rightward, requiring higher latent ability to achieve success.Discrimination Parameter ($a_i$): Dictates the steepness of the logistic curve at $\theta = b_i$.Sequential Likelihood ContributionFor a single observation $y_k \in \{0, 1\}$ at step $k$:$$L(y_k \mid \theta) = [P_k(\theta)]^{y_k} [1 - P_k(\theta)]^{1 - y_k}$$For the running history vector $\mathbf{Y}_k = [y_1, y_2, \dots, y_k]^T$:$$L(\mathbf{Y}_k \mid \theta) = \prod_{i=1}^{k} \left( \frac{1}{1 + e^{-a_i(\theta - b_i)}} \right)^{y_i} \left( 1 - \frac{1}{1 + e^{-a_i(\theta - b_i)}} \right)^{1 - y_i}$$Recursive Update & Dynamic MechanicsThe posterior density at step $k$ satisfies:$$f(\theta \mid \mathbf{Y}_k) = \frac{L(y_k \mid \theta) \, f(\theta \mid \mathbf{Y}_{k-1})}{\int_{-\infty}^{\infty} L(y_k \mid \theta) \, f(\theta \mid \mathbf{Y}_{k-1}) \, d\theta} \propto L(y_k \mid \theta) \, f(\theta \mid \mathbf{Y}_{k-1})$$Dynamic Peak Shifting: When a user correctly answers ($y_k = 1$) an item with large difficulty $b_k$, the likelihood function $P_k(\theta)$ is near zero for low $\theta$ and climbs sharply near $b_k$. Multiplying the prior $f(\theta \mid \mathbf{Y}_{k-1})$ by this monotonically increasing likelihood heavily penalizes lower ability regions, causing a pronounced rightward shift in the peak (MAP) and mean of the posterior.Certainty & Discrimination ($a_k$): The discrimination parameter acts as a precision amplifier. A large $a_k$ creates a sharp step-function likelihood near $b_k$, significantly shrinking posterior variance (increasing sharpness). Conversely, a tiny $a_k$ yields a flat likelihood that barely modifies the prior.Grid Numerical ImplementationBecause the 2PL likelihood is non-conjugate with a Normal prior, updates are tracked on a fixed grid $\boldsymbol{\theta} \in [-6, 6]$:Evaluate $P_k(\theta_j)$ and compute point-wise likelihood $L(y_k \mid \theta_j)$.Compute unnormalized posterior: $f^*_k(\theta_j) = f_{k-1}(\theta_j) \times L(y_k \mid \theta_j)$.Perform numerical integration using the trapezoidal rule: $Z_k = \text{trapz}(f^*_k, \boldsymbol{\theta})$.Normalize: $f_k(\theta_j) = f^*_k(\theta_j) / Z_k$.Point estimates:$$\hat{\theta}_{\text{Bayes}} = \int \theta f_k(\theta) d\theta \approx \text{trapz}(\boldsymbol{\theta} \odot f_k, \boldsymbol{\theta})$$$$\hat{\theta}_{\text{MAP}} = \arg\max_{\theta_j} f_k(\theta_j)$$

# **2. Click-Through Rate Estimation (Beta-Binomial Conjugacy)**

Likelihood & Closed-Form Conjugate ProofFor a single impression $y_k \in \{0, 1\}$ with click probability $\theta$:$$L(y_k \mid \theta) = \theta^{y_k} (1 - \theta)^{1 - y_k}$$For running vector $\mathbf{Y}_k$:$$L(\mathbf{Y}_k \mid \theta) = \theta^{S_k} (1 - \theta)^{k - S_k} \quad \text{where } S_k = \sum_{i=1}^{k} y_i$$Analytical Proof of ConjugacyAssuming prior $f(\theta \mid \mathbf{Y}_{k-1}) = \text{Beta}(\alpha_{k-1}, \beta_{k-1}) \propto \theta^{\alpha_{k-1}-1} (1-\theta)^{\beta_{k-1}-1}$:$$f(\theta \mid \mathbf{Y}_k) \propto L(y_k \mid \theta) f(\theta \mid \mathbf{Y}_{k-1}) = \theta^{y_k} (1-\theta)^{1-y_k} \times \theta^{\alpha_{k-1}-1} (1-\theta)^{\beta_{k-1}-1} = \theta^{(\alpha_{k-1} + y_k) - 1} (1-\theta)^{(\beta_{k-1} + 1 - y_k) - 1}$$Thus, the posterior remains strictly within the Beta family with closed-form updates:$$\alpha_k = \alpha_{k-1} + y_k, \quad \beta_k = \beta_{k-1} + (1 - y_k)$$Closed-Form Point Estimates$$\hat{\theta}_{\text{Bayes}}^{(k)} = \frac{\alpha_k}{\alpha_k + \beta_k} = \frac{\alpha_0 + S_k}{\alpha_0 + \beta_0 + k}$$$$\hat{\theta}_{\text{MAP}}^{(k)} = \frac{\alpha_k - 1}{\alpha_k + \beta_k - 2} \quad (\text{for } \alpha_k, \beta_k > 1)$$Dynamic Behavior vs. IRT Grid UpdatesUnlike IRT (which requires continuous grid evaluation and numerical integration), conjugate Beta updates skip grid approximations entirely. An observed click ($y_k = 1$) increments $\alpha$ directly, pulling the distribution mass toward 1; a non-click ($y_k = 0$) increments $\beta$, pulling mass toward 0. As sample size $k \to \infty$, the prior weight $(\alpha_0 + \beta_0)$ becomes negligible, causing $\hat{\theta}_{\text{Bayes}}$ to converge smoothly to the empirical MLE $\frac{S_k}{k}$.

# **3. Structural Health Monitoring (Bounded Non-Conjugate Updates)**


Prior & Physics-Based LikelihoodThe structural efficiency parameter $\theta$ is bounded to $\theta \in (0, 1]$, initialized via prior $\text{Beta}(8, 2)$ over $(0, 1]$:$$\mathbb{E}[\theta_{\text{prior}}] = \frac{8}{8 + 2} = 0.80$$This prior reflects strong engineering confidence in a healthy baseline component while maintaining smooth boundaries.Given a sensor reading $x_k = \theta \cdot x_{\text{baseline}} \cdot e^{\epsilon_k}$ where $\epsilon_k \sim \mathcal{N}(0, \sigma^2)$, the measurement $x_k$ follows a log-normal distribution conditional on $\theta$:$$L(x_k \mid \theta) = \frac{1}{x_k \sigma \sqrt{2\pi}} \exp\left( -\frac{(\ln x_k - \ln(\theta x_{\text{baseline}}))^2}{2\sigma^2} \right)$$For vector $\mathbf{X}_k = [x_1, \dots, x_k]^T$:$$L(\mathbf{X}_k \mid \theta) = \prod_{i=1}^{k} \frac{1}{x_i \sigma \sqrt{2\pi}} \exp\left( -\frac{(\ln x_i - \ln(\theta x_{\text{baseline}}))^2}{2\sigma^2} \right)$$Non-Conjugate Grid Update FormulationCombining a Beta prior over bounded domain $(0, 1]$ with a Log-Normal likelihood yields no closed-form posterior. The posterior must be calculated numerically on a fine grid $\boldsymbol{\theta} \in [\epsilon, 1]$:$$f(\theta \mid \mathbf{X}_k) = \frac{L(x_k \mid \theta) f(\theta \mid \mathbf{X}_{k-1})}{\int_{0}^{1} L(x_k \mid \theta) f(\theta \mid \mathbf{X}_{k-1}) d\theta}$$Point Estimator Equations:$$\hat{\theta}_{\text{Bayes}}^{(k)} = \frac{\int_{0}^{1} \theta f(\theta \mid \mathbf{X}_k) d\theta}{\int_{0}^{1} f(\theta \mid \mathbf{X}_k) d\theta}$$$$\hat{\theta}_{\text{MAP}}^{(k)} = \arg\max_{\theta \in (0, 1]} f(\theta \mid \mathbf{X}_k)$$

Complete Python Simulation Script
Below is a unified, runnable Python implementation executing the grid update simulation for Structural Health Monitoring (Task 3).

In [1]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

# Set seed for reproducible sensor noise
np.random.seed(42)

# 1. Physical Parameters & Simulation Setup
theta_true = 0.68          # Structural damage state (68% remaining stiffness)
x_baseline = 100.0         # Pristine stiffness baseline (e.g., MN/m)
sigma_noise = 0.15         # Log-space sensor noise standard deviation
n_steps = 30               # Total sequential sensor readings

# Discrete bounded grid for theta in (0, 1]
grid_size = 1000
theta_grid = np.linspace(0.001, 1.0, grid_size)

# 2. Prior Initialization: Beta(8, 2)
alpha_prior, beta_prior = 8.0, 2.0
prior_density = stats.beta.pdf(theta_grid, alpha_prior, beta_prior)

# Ensure proper normalization on bounded grid
prior_density /= np.trapezoid(prior_density, theta_grid)

# 3. Sensor Stream Simulation: Log-Normal Physics Model
# x_k = theta_true * x_baseline * exp(N(0, sigma^2))
sensor_readings = theta_true * x_baseline * np.exp(np.random.normal(0, sigma_noise, size=n_steps))

# 4. Tracking Arrays
bayes_estimates = [np.trapezoid(theta_grid * prior_density, theta_grid)]
map_estimates = [theta_grid[np.argmax(prior_density)]]

milestones = [0, 1, 3, 5, 10, 30]
captured_densities = {0: prior_density.copy()}

current_posterior = prior_density.copy()

# 5. Sequential Numerical Grid Update Loop
for k in range(1, n_steps + 1):
    x_k = sensor_readings[k - 1]

    # Evaluate Log-Normal Likelihood across the grid
    # mu_k = ln(theta * x_baseline)
    mu_grid = np.log(theta_grid * x_baseline)
    likelihood = (1.0 / (x_k * sigma_noise * np.sqrt(2 * np.pi))) * \
                 np.exp(- (np.log(x_k) - mu_grid)**2 / (2 * sigma_noise**2))

    # Recursive update: Posterior proportional to Likelihood * Prior
    current_posterior *= likelihood

    # Sequential Trapezoidal Normalization Step
    norm_constant = np.trapezoid(current_posterior, theta_grid)
    current_posterior /= norm_constant

    # Compute point estimators
    theta_bayes = np.trapezoid(theta_grid * current_posterior, theta_grid)
    theta_map = theta_grid[np.argmax(current_posterior)]

    bayes_estimates.append(theta_bayes)
    map_estimates.append(theta_map)

    if k in milestones:
        captured_densities[k] = current_posterior.copy()

# 6. Visualization 1: Posterior Density Progression Curves
fig1 = go.Figure()
for step, density in captured_densities.items():
    fig1.add_trace(go.Scatter(
        x=theta_grid, y=density,
        mode='lines',
        name=f"Step {step}" if step > 0 else "Initial Prior Beta(8,2)",
        line=dict(width=2.5 if step in [0, n_steps] else 1.5)
    ))

fig1.add_vline(
    x=theta_true, line_dash="dash", line_color="red", line_width=2,
    annotation_text=f"True Stiffness (θ = {theta_true})", annotation_position="top left"
)

fig1.update_layout(
    title="Structural Health Monitoring: Posterior Density Progression",
    xaxis_title="Stiffness Efficiency Factor (θ)",
    yaxis_title="Probability Density f(θ | X_k)",
    template="plotly_white",
    hovermode="x unified"
)
fig1.show()

# 7. Visualization 2: Convergence Timeline
steps_axis = list(range(n_steps + 1))
fig2 = go.Figure()

fig2.add_hline(
    y=theta_true, line_dash="dash", line_color="red", line_width=2,
    annotation_text=f"True Efficiency (θ = {theta_true})"
)
fig2.add_trace(go.Scatter(
    x=steps_axis, y=bayes_estimates,
    mode='lines+markers', name='Posterior Mean (θ̂_Bayes)',
    line=dict(color='blue', width=2.5)
))
fig2.add_trace(go.Scatter(
    x=steps_axis, y=map_estimates,
    mode='lines+markers', name='MAP Estimate (θ̂_MAP)',
    line=dict(color='green', width=2, dash='dot')
))

fig2.update_layout(
    title="Structural Health Monitoring: Parameter Estimation Convergence",
    xaxis_title="Inspection Step (k)",
    yaxis_title="Estimated Efficiency (θ̂)",
    template="plotly_white",
    hovermode="x unified"
)
fig2.show()

# **SHM Convergence Analysis**

Overcoming Optimistic Prior: The initial prior $\text{Beta}(8,2)$ strongly favors a healthy state ($\mathbb{E}[\theta] = 0.80$). Within 3 to 5 sensor observations, the log-normal likelihood contribution overwhelms the prior, causing both $\hat{\theta}_{\text{Bayes}}$ and $\hat{\theta}_{\text{MAP}}$ to rapidly shift toward the true damage parameter $\theta = 0.68$.Structural Safety Implications: As $k$ increases, the posterior distribution narrows significantly around $\theta = 0.68$. This variance reduction corresponds directly to high confidence in structural degradation, allowing automated safety engines to flag critical thresholds (e.g., $\theta < 0.70$) and trigger maintenance protocols with minimal risk of false alarms.